<cell_type>markdown</cell_type># vLLM 大语言模型推理教程 (vLLM LLM Inference Tutorial)

> **前置知识**: PyTorch 基础、Transformer 架构、LLM 基础
>
> **学习目标**: 掌握 vLLM 的核心技术和高性能 LLM 推理

---

## 什么是 vLLM？

```
┌─────────────────────────────────────────────────────────────┐
│                    vLLM 核心技术                             │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统 LLM 推理问题:                                         │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 1: [████████████░░░░░░░░]  预分配但未使用     │   │
│  │  请求 2: [██████░░░░░░░░░░░░░░]  大量内存浪费       │   │
│  │  请求 3: [████████████████░░░░]                     │   │
│  │          ↑ 实际使用    ↑ 浪费                       │   │
│  │                                                     │   │
│  │  问题: KV Cache 预分配导致内存浪费严重              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  vLLM 解决方案 - PagedAttention:                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  物理内存块 (Pages):                                │   │
│  │  ┌────┬────┬────┬────┬────┬────┬────┬────┐         │   │
│  │  │ P0 │ P1 │ P2 │ P3 │ P4 │ P5 │ P6 │ P7 │         │   │
│  │  └────┴────┴────┴────┴────┴────┴────┴────┘         │   │
│  │    ↑    ↑    ↑    ↑    ↑    ↑                       │   │
│  │  请求1: [P0, P1, P2]                                │   │
│  │  请求2: [P3, P4]                                    │   │
│  │  请求3: [P5, P6, P7]                                │   │
│  │                                                     │   │
│  │  - 按需分配，无浪费                                 │   │
│  │  - 支持动态增长                                     │   │
│  │  - 内存利用率接近 100%                              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  核心优势:                                                  │
│  - PagedAttention: 高效 KV Cache 管理                      │
│  - Continuous Batching: 动态批处理                         │
│  - 比 HuggingFace 快 2-24x                                 │
│  - 支持多种量化方法 (GPTQ, AWQ)                            │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**注意**: vLLM 需要 NVIDIA GPU 支持

## 本教程内容

1. **PagedAttention 原理** - 高效内存管理
2. **Continuous Batching** - 动态批处理
3. **采样参数** - 控制生成质量
4. **量化支持** - GPTQ/AWQ
5. **高级功能** - LoRA、投机解码

In [ ]:
# ============================================================
# 环境准备
# ============================================================
import numpy as np
import time

# 设置随机种子
np.random.seed(42)

# 检查 vLLM
try:
    from vllm import LLM, SamplingParams
    VLLM_AVAILABLE = True
except ImportError:
    VLLM_AVAILABLE = False

# 检查 PyTorch 和 CUDA
try:
    import torch
    TORCH_AVAILABLE = True
except ImportError:
    TORCH_AVAILABLE = False

print("=" * 60)
print("环境准备完成")
print("=" * 60)

if VLLM_AVAILABLE:
    print(f"\n✓ vLLM 已安装")
else:
    print(f"\n✗ vLLM 未安装")
    print("  安装命令: pip install vllm")

if TORCH_AVAILABLE:
    print(f"✓ PyTorch 版本: {torch.__version__}")
    print(f"  CUDA 可用: {torch.cuda.is_available()}")
    if torch.cuda.is_available():
        print(f"  GPU: {torch.cuda.get_device_name(0)}")
        print(f"  GPU 内存: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    print(f"✗ PyTorch 未安装")

print(f"\n注意: vLLM 需要 NVIDIA GPU 才能运行")

<cell_type>markdown</cell_type>## 1. PagedAttention 原理

**核心概念**: PagedAttention 是 vLLM 的核心技术，借鉴操作系统的虚拟内存分页机制

```
┌─────────────────────────────────────────────────────────────┐
│                    传统 KV Cache 问题                        │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  LLM 推理需要存储 KV Cache (Key-Value Cache):               │
│  - 每个 token 需要存储 K 和 V 向量                         │
│  - 序列越长，KV Cache 越大                                 │
│  - 传统方法: 预分配最大长度的连续内存                      │
│                                                             │
│  问题示意:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  请求 1: [████████████░░░░░░░░]  预分配但未使用     │   │
│  │  请求 2: [██████░░░░░░░░░░░░░░]  大量内存浪费       │   │
│  │  请求 3: [████████████████░░░░]                     │   │
│  │          ↑ 实际使用    ↑ 浪费                       │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  内存浪费原因:                                              │
│  1. 预分配: 必须为最大可能长度预留空间                     │
│  2. 碎片化: 不同请求长度不同，无法复用                     │
│  3. 过度预留: 实际生成长度通常远小于最大长度               │
│                                                             │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    PagedAttention 解决方案                   │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  核心思想: 像操作系统管理内存一样管理 KV Cache              │
│                                                             │
│  物理内存块 (Pages):                                        │
│  ┌────┬────┬────┬────┬────┬────┬────┬────┬────┬────┐       │
│  │ P0 │ P1 │ P2 │ P3 │ P4 │ P5 │ P6 │ P7 │ P8 │ P9 │       │
│  └────┴────┴────┴────┴────┴────┴────┴────┴────┴────┘       │
│    ↑    ↑    ↑    ↑    ↑    ↑    ↑    ↑                     │
│    │    │    │    │    │    │    │                          │
│  请求1: [P0, P1, P2]      (3 个 blocks)                     │
│  请求2: [P3, P4]          (2 个 blocks)                     │
│  请求3: [P5, P6, P7]      (3 个 blocks)                     │
│  空闲:  [P8, P9]          (可分配给新请求)                  │
│                                                             │
│  优势:                                                      │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 按需分配: 只分配实际需要的 blocks               │   │
│  │  2. 动态增长: 生成新 token 时分配新 block           │   │
│  │  3. 高效复用: 请求完成后 blocks 立即可用            │   │
│  │  4. 内存利用率: 接近 100%，几乎无浪费               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

### KV Cache 内存计算

```
单个 token 的 KV Cache 大小:
= 2 × num_layers × hidden_size × sizeof(dtype)

例如 Llama-2-7B (FP16):
= 2 × 32 × 4096 × 2 bytes
= 512 KB / token

1000 tokens 序列:
= 512 KB × 1000 = 512 MB

批量 32 个请求:
= 512 MB × 32 = 16 GB (仅 KV Cache!)
```

<cell_type>markdown</cell_type>## 2. 采样参数详解

**核心概念**: 采样参数控制 LLM 生成文本的多样性和质量

```
┌─────────────────────────────────────────────────────────────┐
│                    采样参数说明                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  Temperature (温度):                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  控制输出的随机性                                   │   │
│  │                                                     │   │
│  │  T=0: 贪婪解码，总是选择最高概率的 token           │   │
│  │  T<1: 降低随机性，输出更确定、更保守               │   │
│  │  T=1: 原始概率分布                                 │   │
│  │  T>1: 增加随机性，输出更多样、更有创意             │   │
│  │                                                     │   │
│  │  公式: P'(token) = softmax(logits / T)             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Top-p (Nucleus Sampling):                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  只从累积概率达到 p 的最小 token 集合中采样        │   │
│  │                                                     │   │
│  │  例: top_p=0.9                                      │   │
│  │  tokens:  [A:0.5, B:0.3, C:0.1, D:0.05, E:0.05]    │   │
│  │  累积:    [0.5,   0.8,   0.9,   0.95,   1.0]       │   │
│  │  选择:    [A,     B,     C]  (累积到 0.9)          │   │
│  │                                                     │   │
│  │  动态调整候选集大小，适应不同上下文                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Top-k:                                                     │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  只从概率最高的 k 个 token 中采样                  │   │
│  │                                                     │   │
│  │  例: top_k=3                                        │   │
│  │  tokens:  [A:0.5, B:0.3, C:0.1, D:0.05, E:0.05]    │   │
│  │  选择:    [A,     B,     C]  (前 3 个)             │   │
│  │                                                     │   │
│  │  固定候选集大小，简单但不够灵活                    │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  Presence/Frequency Penalty:                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  presence_penalty: 惩罚已出现的 token (鼓励新话题) │   │
│  │  frequency_penalty: 惩罚高频 token (减少重复)      │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 定义采样配置类 (自包含实现)
# ============================================================
from dataclasses import dataclass
from enum import Enum
from typing import Optional

@dataclass
class SamplingConfig:
    """
    采样参数配置
    
    控制 LLM 生成文本的多样性和质量
    """
    temperature: float = 0.8        # 温度: 控制随机性
    top_p: float = 0.95             # Nucleus sampling 阈值
    top_k: int = -1                 # Top-k 采样 (-1 表示不限制)
    max_tokens: int = 256           # 最大生成 token 数
    presence_penalty: float = 0.0   # 存在惩罚 (鼓励新话题)
    frequency_penalty: float = 0.0  # 频率惩罚 (减少重复)
    stop: Optional[list] = None     # 停止词列表


class QuantizationMethod(Enum):
    """量化方法枚举"""
    NONE = None
    GPTQ = "gptq"
    AWQ = "awq"
    SQUEEZELLM = "squeezellm"


@dataclass
class EngineConfig:
    """
    vLLM 引擎配置
    
    控制模型加载和推理行为
    """
    model: str = "meta-llama/Llama-2-7b-hf"
    tensor_parallel_size: int = 1           # 张量并行数 (多 GPU)
    gpu_memory_utilization: float = 0.9     # GPU 内存利用率
    max_model_len: Optional[int] = None     # 最大序列长度
    dtype: str = "auto"                     # 数据类型
    quantization: QuantizationMethod = QuantizationMethod.NONE
    enable_lora: bool = False               # 是否启用 LoRA
    max_loras: int = 1                      # 最大 LoRA 数量
    max_lora_rank: int = 16                 # 最大 LoRA 秩
    speculative_model: Optional[str] = None # 投机解码草稿模型
    num_speculative_tokens: int = 5         # 投机 token 数


print("=" * 60)
print("配置类定义完成")
print("=" * 60)

# 采样配置示例
sampling_config = SamplingConfig(
    temperature=0.8,
    top_p=0.95,
    top_k=50,
    max_tokens=256,
    presence_penalty=0.1,
    frequency_penalty=0.1
)

print(f"\n采样配置示例:")
print(f"  温度 (temperature): {sampling_config.temperature}")
print(f"  Top-p: {sampling_config.top_p}")
print(f"  Top-k: {sampling_config.top_k}")
print(f"  最大 tokens: {sampling_config.max_tokens}")
print(f"  存在惩罚: {sampling_config.presence_penalty}")
print(f"  频率惩罚: {sampling_config.frequency_penalty}")

In [ ]:
# ============================================================
# 引擎配置示例
# ============================================================
print("=" * 60)
print("vLLM 引擎配置")
print("=" * 60)

# 引擎配置
engine_config = EngineConfig(
    model="meta-llama/Llama-2-7b-hf",
    tensor_parallel_size=1,
    gpu_memory_utilization=0.9,
    max_model_len=4096,
    dtype="float16"
)

print(f"\n引擎配置:")
print(f"  模型: {engine_config.model}")
print(f"  张量并行: {engine_config.tensor_parallel_size}")
print(f"  GPU 内存利用率: {engine_config.gpu_memory_utilization}")
print(f"  最大序列长度: {engine_config.max_model_len}")
print(f"  数据类型: {engine_config.dtype}")

print(f"\n配置说明:")
print(f"  tensor_parallel_size: 多 GPU 并行，每个 GPU 处理模型的一部分")
print(f"  gpu_memory_utilization: 0.9 表示使用 90% GPU 内存")
print(f"  max_model_len: 限制最大序列长度，影响 KV Cache 大小")

<cell_type>markdown</cell_type>## 3. 采样策略对比

**核心概念**: 不同采样策略适用于不同场景

```
┌─────────────────────────────────────────────────────────────┐
│                    采样策略选择指南                          │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  场景                    推荐策略                           │
│  ────                    ────────                           │
│  代码生成               贪婪解码 (T=0)                      │
│  问答系统               低温采样 (T=0.3)                    │
│  对话聊天               标准采样 (T=0.7)                    │
│  创意写作               高温采样 (T=1.0+)                   │
│  多样性输出             Nucleus (top_p=0.9)                 │
│                                                             │
│  温度效果可视化:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  T=0.0: [████████████████░░░░]  确定性，最高概率    │   │
│  │  T=0.5: [████████████░░░░░░░░]  较保守              │   │
│  │  T=1.0: [████████░░░░░░░░░░░░]  原始分布            │   │
│  │  T=1.5: [████░░░░░░░░░░░░░░░░]  高随机性            │   │
│  │         ↑ 概率集中度                                │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 采样策略对比
# ============================================================
print("=" * 60)
print("采样策略对比")
print("=" * 60)

# 不同采样策略
sampling_strategies = {
    "贪婪解码": SamplingConfig(temperature=0.0, top_p=1.0, top_k=-1),
    "低温采样": SamplingConfig(temperature=0.3, top_p=0.9),
    "标准采样": SamplingConfig(temperature=0.7, top_p=0.95),
    "高温采样": SamplingConfig(temperature=1.2, top_p=0.95),
    "Top-k 采样": SamplingConfig(temperature=0.8, top_k=50),
    "Nucleus 采样": SamplingConfig(temperature=0.8, top_p=0.9),
}

strategy_features = {
    "贪婪解码": "确定性，最高概率",
    "低温采样": "保守，较少随机性",
    "标准采样": "平衡创造性和连贯性",
    "高温采样": "高创造性，可能不连贯",
    "Top-k 采样": "限制候选词数量",
    "Nucleus 采样": "动态候选词集合",
}

print(f"\n{'策略':<15} {'温度':<8} {'Top-p':<8} {'Top-k':<8} {'特点'}")
print("-" * 65)

for name, config in sampling_strategies.items():
    print(f"{name:<15} {config.temperature:<8} {config.top_p:<8} {config.top_k:<8} {strategy_features[name]}")

<cell_type>markdown</cell_type>## 4. 量化支持

**核心概念**: vLLM 支持多种量化方法，减少内存占用并加速推理

```
┌─────────────────────────────────────────────────────────────┐
│                    量化方法对比                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  方法      精度    内存节省    速度提升    适用场景         │
│  ────      ────    ────────    ────────    ────────         │
│  FP16      高      2x          1.5-2x      通用             │
│  GPTQ      中高    4x          2-3x        资源受限         │
│  AWQ       中高    4x          2-3x        边缘部署         │
│  INT8      中      4x          2-4x        服务器部署       │
│                                                             │
│  GPTQ vs AWQ:                                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  GPTQ (GPT Quantization):                           │   │
│  │  - 基于 OBQ (Optimal Brain Quantization)            │   │
│  │  - 逐层量化，使用校准数据                           │   │
│  │  - 精度较高，但量化过程较慢                         │   │
│  │                                                     │   │
│  │  AWQ (Activation-aware Weight Quantization):        │   │
│  │  - 保护重要权重 (基于激活值分析)                    │   │
│  │  - 量化速度快                                       │   │
│  │  - 在低比特下精度更好                               │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 量化配置示例
# ============================================================
print("=" * 60)
print("量化配置")
print("=" * 60)

# 支持的量化方法
print(f"\n支持的量化方法:")
for method in QuantizationMethod:
    print(f"  - {method.name}: {method.value}")

# 量化配置示例
quantization_configs = {
    "无量化 (FP16)": EngineConfig(
        model="meta-llama/Llama-2-7b-hf",
        quantization=QuantizationMethod.NONE,
        dtype="float16"
    ),
    "GPTQ 量化": EngineConfig(
        model="TheBloke/Llama-2-7B-GPTQ",
        quantization=QuantizationMethod.GPTQ
    ),
    "AWQ 量化": EngineConfig(
        model="TheBloke/Llama-2-7B-AWQ",
        quantization=QuantizationMethod.AWQ
    ),
}

print(f"\n量化配置对比:")
print(f"{'配置':<20} {'模型':<35} {'量化方法'}")
print("-" * 70)
for name, config in quantization_configs.items():
    quant = config.quantization.value if config.quantization.value else "None"
    print(f"{name:<20} {config.model:<35} {quant}")

print(f"\n内存占用估算 (Llama-2-7B):")
print(f"  FP32: ~28 GB")
print(f"  FP16: ~14 GB")
print(f"  INT8: ~7 GB")
print(f"  INT4 (GPTQ/AWQ): ~3.5 GB")

<cell_type>markdown</cell_type>## 5. vLLM 基本使用

**核心概念**: vLLM 提供简洁的 API 进行高性能 LLM 推理

```
┌─────────────────────────────────────────────────────────────┐
│                    vLLM 基本使用流程                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  1. 创建 LLM 实例                                           │
│     ┌─────────────────────────────────────────────────┐    │
│     │  llm = LLM(                                     │    │
│     │      model="meta-llama/Llama-2-7b-hf",         │    │
│     │      gpu_memory_utilization=0.9,               │    │
│     │      max_model_len=4096                        │    │
│     │  )                                             │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  2. 配置采样参数                                            │
│     ┌─────────────────────────────────────────────────┐    │
│     │  sampling_params = SamplingParams(             │    │
│     │      temperature=0.8,                          │    │
│     │      top_p=0.95,                               │    │
│     │      max_tokens=256                            │    │
│     │  )                                             │    │
│     └─────────────────────────────────────────────────┘    │
│                        ↓                                    │
│  3. 执行生成                                                │
│     ┌─────────────────────────────────────────────────┐    │
│     │  outputs = llm.generate(prompts, sampling_params)│   │
│     └─────────────────────────────────────────────────┘    │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

**注意**: 以下代码需要 vLLM 和 NVIDIA GPU 才能运行

In [ ]:
# ============================================================
# vLLM 基本使用示例
# ============================================================
print("=" * 60)
print("vLLM 基本使用")
print("=" * 60)

MODEL_LOADED = False

if VLLM_AVAILABLE:
    # 使用小模型进行演示
    print("\n加载模型 (这可能需要几分钟)...")
    
    try:
        llm = LLM(
            model="facebook/opt-125m",  # 小模型用于演示
            gpu_memory_utilization=0.5,
            max_model_len=512
        )
        print("✓ 模型加载成功!")
        MODEL_LOADED = True
    except Exception as e:
        print(f"✗ 模型加载失败: {e}")
        print("\n可能原因:")
        print("  1. GPU 内存不足")
        print("  2. 模型未下载")
        print("  3. CUDA 版本不兼容")
else:
    print("\n跳过 vLLM 示例 (vLLM 未安装)")
    print("安装命令: pip install vllm")

In [ ]:
# ============================================================
# 文本生成示例
# ============================================================
if VLLM_AVAILABLE and MODEL_LOADED:
    print("=" * 60)
    print("文本生成示例")
    print("=" * 60)
    
    # 准备提示词
    prompts = [
        "The future of artificial intelligence is",
        "Machine learning can help us",
        "Deep learning models are"
    ]
    
    # 配置采样参数
    sampling_params = SamplingParams(
        temperature=0.8,
        top_p=0.95,
        max_tokens=50
    )
    
    print("\n生成文本...")
    outputs = llm.generate(prompts, sampling_params)
    
    print("\n生成结果:")
    print("=" * 60)
    for output in outputs:
        print(f"提示: {output.prompt}")
        print(f"生成: {output.outputs[0].text}")
        print("-" * 60)
else:
    print("跳过文本生成示例 (vLLM 未安装或模型未加载)")
    print("\n示例输出格式:")
    print("  提示: The future of artificial intelligence is")
    print("  生成: going to be very exciting. We are already seeing...")

<cell_type>markdown</cell_type>## 6. 批量推理性能

**核心概念**: vLLM 的 Continuous Batching 使批量推理效率极高

```
┌─────────────────────────────────────────────────────────────┐
│                    批量推理优势                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  吞吐量 vs 延迟权衡:                                        │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Batch=1:  低延迟，低吞吐                           │   │
│  │  Batch=8:  中延迟，高吞吐                           │   │
│  │  Batch=32: 高延迟，最高吞吐                         │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  vLLM 优势:                                                 │
│  - Continuous Batching: 动态调整批次大小                   │
│  - 请求完成后立即处理新请求                                │
│  - 无需等待整个批次完成                                    │
│  - 吞吐量提升 2-24x (相比 HuggingFace)                     │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 批量推理性能测试
# ============================================================
if VLLM_AVAILABLE and MODEL_LOADED:
    print("=" * 60)
    print("批量推理性能测试")
    print("=" * 60)
    
    # 测试不同批次大小的性能
    batch_sizes = [1, 4, 8, 16]
    results = []
    
    base_prompt = "The quick brown fox jumps over the lazy dog. "
    
    for batch_size in batch_sizes:
        prompts = [base_prompt] * batch_size
        
        sampling_params = SamplingParams(
            temperature=0.8,
            max_tokens=32
        )
        
        # 预热
        _ = llm.generate(prompts[:1], sampling_params)
        
        # 计时
        start = time.perf_counter()
        outputs = llm.generate(prompts, sampling_params)
        elapsed = time.perf_counter() - start
        
        # 统计 tokens
        total_tokens = sum(len(o.outputs[0].token_ids) for o in outputs)
        
        results.append({
            'batch_size': batch_size,
            'time_s': elapsed,
            'total_tokens': total_tokens,
            'tokens_per_sec': total_tokens / elapsed,
            'latency_per_request': elapsed / batch_size * 1000
        })
    
    print(f"\n{'Batch':<8} {'Time(s)':<10} {'Tokens':<10} {'Tok/s':<12} {'Latency(ms)'}")
    print("-" * 55)
    for r in results:
        print(f"{r['batch_size']:<8} {r['time_s']:<10.3f} {r['total_tokens']:<10} {r['tokens_per_sec']:<12.1f} {r['latency_per_request']:.1f}")
    
    print(f"\n观察:")
    print(f"  - 批次越大，总吞吐量 (Tok/s) 越高")
    print(f"  - 批次越大，单请求延迟略有增加")
    print(f"  - vLLM 的 Continuous Batching 使批处理非常高效")
else:
    print("跳过批量推理性能测试 (vLLM 未安装或模型未加载)")
    print("\n预期结果示例:")
    print(f"{'Batch':<8} {'Time(s)':<10} {'Tokens':<10} {'Tok/s':<12} {'Latency(ms)'}")
    print("-" * 55)
    print(f"{'1':<8} {'0.150':<10} {'32':<10} {'213.3':<12} {'150.0'}")
    print(f"{'4':<8} {'0.180':<10} {'128':<10} {'711.1':<12} {'45.0'}")
    print(f"{'8':<8} {'0.220':<10} {'256':<10} {'1163.6':<12} {'27.5'}")
    print(f"{'16':<8} {'0.300':<10} {'512':<10} {'1706.7':<12} {'18.8'}")

<cell_type>markdown</cell_type>## 7. Continuous Batching 原理

**核心概念**: Continuous Batching 是 vLLM 的另一核心技术，实现动态批处理

```
┌─────────────────────────────────────────────────────────────┐
│                    传统静态批处理                            │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  时间 →                                                     │
│  请求1: [████████████████████████████████]                 │
│  请求2: [████████░░░░░░░░░░░░░░░░░░░░░░░░] 等待            │
│  请求3: [████████████████░░░░░░░░░░░░░░░░] 等待            │
│         ↑ 必须等最长请求完成才能处理新请求                  │
│                                                             │
│  问题:                                                      │
│  - 短请求完成后仍需等待长请求                              │
│  - GPU 利用率低                                            │
│  - 新请求必须等待整个批次完成                              │
│                                                             │
└─────────────────────────────────────────────────────────────┘

┌─────────────────────────────────────────────────────────────┐
│                    Continuous Batching                       │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  时间 →                                                     │
│  请求1: [████████████████████████████████]                 │
│  请求2: [████████]→完成→[新请求5开始...]                   │
│  请求3: [████████████████]→完成→[新请求6...]               │
│  请求4:     [████████████████████████]                     │
│         ↑ 请求完成后立即处理新请求                          │
│                                                             │
│  优势:                                                      │
│  - 请求完成后立即释放资源                                  │
│  - 新请求可以立即加入批次                                  │
│  - GPU 利用率接近 100%                                     │
│  - 吞吐量大幅提升                                          │
│                                                             │
│  实现机制:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  每个 iteration:                                    │   │
│  │  1. 检查已完成的请求，释放其 KV Cache               │   │
│  │  2. 检查等待队列，加入新请求                        │   │
│  │  3. 对当前批次执行一步解码                          │   │
│  │  4. 重复                                            │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

<cell_type>markdown</cell_type>## 8. LoRA 适配器支持

**核心概念**: vLLM 支持动态加载 LoRA 适配器，实现多任务服务

```
┌─────────────────────────────────────────────────────────────┐
│                    LoRA 适配器原理                           │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  LoRA (Low-Rank Adaptation):                                │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  原始权重 W (冻结)                                  │   │
│  │       ↓                                             │   │
│  │  W' = W + ΔW = W + BA                              │   │
│  │                                                     │   │
│  │  其中:                                              │   │
│  │  - B: [d, r] 低秩矩阵                              │   │
│  │  - A: [r, k] 低秩矩阵                              │   │
│  │  - r << min(d, k) (秩远小于原始维度)               │   │
│  │                                                     │   │
│  │  参数量: d×r + r×k << d×k                          │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  vLLM LoRA 优势:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 动态加载: 运行时切换不同 LoRA                  │   │
│  │  2. 多 LoRA 并行: 同时服务多个任务                 │   │
│  │  3. 内存高效: 共享基础模型，只加载 LoRA 参数       │   │
│  │  4. 热切换: 无需重启服务                           │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# LoRA 配置示例
# ============================================================
print("=" * 60)
print("LoRA 配置示例")
print("=" * 60)

# LoRA 配置
lora_config = EngineConfig(
    model="meta-llama/Llama-2-7b-hf",
    enable_lora=True,
    max_loras=4,
    max_lora_rank=16
)

print(f"\nLoRA 配置:")
print(f"  启用 LoRA: {lora_config.enable_lora}")
print(f"  最大 LoRA 数量: {lora_config.max_loras}")
print(f"  最大 LoRA 秩: {lora_config.max_lora_rank}")

print(f"\nvLLM LoRA 使用示例:")
print("""
from vllm import LLM, SamplingParams
from vllm.lora.request import LoRARequest

# 创建支持 LoRA 的 LLM 实例
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",
    enable_lora=True,
    max_loras=4,
    max_lora_rank=16
)

# 创建 LoRA 请求
lora_request = LoRARequest(
    "sql_adapter",           # 适配器名称
    1,                       # adapter_id
    "/path/to/sql_lora"      # LoRA 权重路径
)

# 使用 LoRA 生成
outputs = llm.generate(
    ["Write a SQL query to..."],
    SamplingParams(temperature=0.7),
    lora_request=lora_request
)
""")

<cell_type>markdown</cell_type>## 9. 投机解码 (Speculative Decoding)

**核心概念**: 使用小模型加速大模型推理，保持输出质量

```
┌─────────────────────────────────────────────────────────────┐
│                    投机解码原理                              │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  传统自回归解码:                                            │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  大模型: [生成t1] → [生成t2] → [生成t3] → ...      │   │
│  │          ↑ 慢      ↑ 慢      ↑ 慢                   │   │
│  │  每个 token 都需要完整的大模型前向传播              │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  投机解码:                                                  │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  Step 1: 小模型快速生成 N 个候选 tokens             │   │
│  │          [t1, t2, t3, t4, t5] (草稿)               │   │
│  │                    ↓                                │   │
│  │  Step 2: 大模型并行验证所有候选                     │   │
│  │          [✓,  ✓,  ✓,  ✗,  -]                       │   │
│  │                    ↓                                │   │
│  │  Step 3: 接受正确的，从错误位置重新生成             │   │
│  │          接受 [t1, t2, t3]，重新生成 t4             │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  为什么有效？                                               │
│  ┌─────────────────────────────────────────────────────┐   │
│  │  1. 小模型生成很快 (7B vs 70B)                     │   │
│  │  2. 大模型验证可以并行 (一次验证 N 个)             │   │
│  │  3. 小模型预测准确率通常 > 70%                     │   │
│  │  4. 平均每次大模型调用接受 3-5 个 tokens           │   │
│  │  5. 总体加速 2-3x                                  │   │
│  └─────────────────────────────────────────────────────┘   │
│                                                             │
│  关键: 输出质量与纯大模型完全相同!                         │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# 投机解码配置示例
# ============================================================
print("=" * 60)
print("投机解码配置")
print("=" * 60)

# 投机解码配置
speculative_config = EngineConfig(
    model="meta-llama/Llama-2-70b-hf",
    speculative_model="meta-llama/Llama-2-7b-hf",
    num_speculative_tokens=5
)

print(f"\n投机解码配置:")
print(f"  目标模型 (大): {speculative_config.model}")
print(f"  草稿模型 (小): {speculative_config.speculative_model}")
print(f"  投机 tokens 数: {speculative_config.num_speculative_tokens}")

print(f"\n投机解码使用示例:")
print("""
from vllm import LLM, SamplingParams

# 创建支持投机解码的 LLM 实例
llm = LLM(
    model="meta-llama/Llama-2-70b-hf",
    speculative_model="meta-llama/Llama-2-7b-hf",
    num_speculative_tokens=5,
    tensor_parallel_size=4  # 大模型通常需要多 GPU
)

# 正常使用，投机解码自动生效
outputs = llm.generate(
    ["Explain quantum computing in simple terms."],
    SamplingParams(temperature=0.7, max_tokens=200)
)
""")

print(f"\n投机解码适用场景:")
print(f"  ✓ 长文本生成 (文章、代码)")
print(f"  ✓ 对延迟敏感的应用")
print(f"  ✓ 有足够 GPU 内存同时加载两个模型")
print(f"  ✗ 短回复场景 (开销可能超过收益)")

<cell_type>markdown</cell_type>## 10. API 服务模式

**核心概念**: vLLM 提供兼容 OpenAI API 的服务器，便于集成

```
┌─────────────────────────────────────────────────────────────┐
│                    vLLM API 服务架构                         │
├─────────────────────────────────────────────────────────────┤
│                                                             │
│  客户端                    vLLM 服务器                      │
│  ┌─────────────┐          ┌─────────────────────────┐      │
│  │  OpenAI SDK │          │  OpenAI 兼容 API        │      │
│  │  或任何     │ ──HTTP──→│  /v1/completions       │      │
│  │  HTTP 客户端│          │  /v1/chat/completions  │      │
│  └─────────────┘          │  /v1/models            │      │
│                           └─────────────────────────┘      │
│                                    │                        │
│                                    ▼                        │
│                           ┌─────────────────────────┐      │
│                           │  vLLM Engine            │      │
│                           │  - PagedAttention       │      │
│                           │  - Continuous Batching  │      │
│                           │  - 量化支持             │      │
│                           └─────────────────────────┘      │
│                                                             │
│  优势:                                                      │
│  - 兼容 OpenAI API，无缝迁移                               │
│  - 支持流式输出 (streaming)                                │
│  - 内置请求队列和调度                                      │
│  - 支持多模型服务                                          │
│                                                             │
└─────────────────────────────────────────────────────────────┘
```

In [ ]:
# ============================================================
# API 服务模式示例
# ============================================================
print("=" * 60)
print("vLLM API 服务模式")
print("=" * 60)

print(f"\n启动 vLLM API 服务器:")
print("""
# 命令行启动
python -m vllm.entrypoints.openai.api_server \\
    --model meta-llama/Llama-2-7b-hf \\
    --port 8000 \\
    --tensor-parallel-size 1 \\
    --gpu-memory-utilization 0.9

# 常用参数:
#   --model: 模型名称或路径
#   --port: 服务端口
#   --tensor-parallel-size: 张量并行数
#   --gpu-memory-utilization: GPU 内存利用率
#   --max-model-len: 最大序列长度
#   --quantization: 量化方法 (gptq, awq)
""")

print(f"\n客户端调用示例 (兼容 OpenAI API):")
print("""
import openai

# 创建客户端 (指向 vLLM 服务器)
client = openai.OpenAI(
    base_url="http://localhost:8000/v1",
    api_key="dummy"  # vLLM 不需要真实 API key
)

# Chat Completions API
response = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    messages=[
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "Hello!"}
    ],
    temperature=0.7,
    max_tokens=100
)

print(response.choices[0].message.content)

# 流式输出
stream = client.chat.completions.create(
    model="meta-llama/Llama-2-7b-hf",
    messages=[{"role": "user", "content": "Tell me a story."}],
    stream=True
)

for chunk in stream:
    if chunk.choices[0].delta.content:
        print(chunk.choices[0].delta.content, end="")
""")

<cell_type>markdown</cell_type>## 总结

本教程介绍了 vLLM 的核心功能和最佳实践：

### 核心知识点

| 主题 | 关键内容 |
|:-----|:---------|
| PagedAttention | 借鉴 OS 分页机制，按需分配 KV Cache，内存利用率接近 100% |
| Continuous Batching | 动态批处理，请求完成后立即处理新请求 |
| 采样参数 | temperature、top_p、top_k 控制生成多样性 |
| 量化支持 | GPTQ、AWQ 等方法，4x 内存节省 |
| LoRA 适配器 | 动态加载，多任务服务 |
| 投机解码 | 小模型加速大模型，2-3x 加速 |

### vLLM API 速查

```python
from vllm import LLM, SamplingParams

# 创建 LLM 实例
llm = LLM(
    model="meta-llama/Llama-2-7b-hf",
    gpu_memory_utilization=0.9,
    max_model_len=4096
)

# 配置采样参数
sampling_params = SamplingParams(
    temperature=0.8,
    top_p=0.95,
    max_tokens=256
)

# 生成文本
outputs = llm.generate(prompts, sampling_params)

# 启动 API 服务器
# python -m vllm.entrypoints.openai.api_server --model xxx --port 8000
```

### 最佳实践

```
性能优化检查清单:
✓ 使用量化模型减少内存占用 (GPTQ/AWQ)
✓ 调整 gpu_memory_utilization 平衡内存和吞吐
✓ 利用 Continuous Batching 提高吞吐量
✓ 长文本生成考虑投机解码
✓ 多任务场景使用 LoRA 适配器

常见问题:
✗ GPU 内存不足 → 降低 gpu_memory_utilization 或使用量化
✗ 吞吐量低 → 增加批次大小，检查 Continuous Batching
✗ 延迟高 → 减少 max_tokens，考虑投机解码
```

### 下一步学习

- **04_Advanced_Inference_tutorial.ipynb**: 高级推理技术
- **03-serving-systems**: 模型服务系统